# ch03 假设检验与效应量

执行比例 Z 检验，计算效应量和统计功效。

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import Config
from src.utils.data_loader import DataLoader
from src.utils.output_manager import OutputManager

print("模块加载成功")

## Step 1: 加载数据

In [ ]:
config = Config()
loader = DataLoader(config)
output = OutputManager(config)

df = loader.load_processed("cleaned_data.csv")
print(f"数据形状: {df.shape}")

## Step 2: 比例 Z 检验（单尾）

In [ ]:
# 提取两组数据
con_data = df[df['group'] == 'con']['click']
exp_data = df[df['group'] == 'exp']['click']

n_con, n_exp = len(con_data), len(exp_data)
x_con, x_exp = con_data.sum(), exp_data.sum()
p_con, p_exp = x_con / n_con, x_exp / n_exp

# 合并比例
p_pooled = (x_con + x_exp) / (n_con + n_exp)

# 计算标准误
se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_con + 1/n_exp))

# Z 统计量
z_stat = (p_exp - p_con) / se

# 单尾 p 值
p_value = 1 - stats.norm.cdf(z_stat)

print(f"对照组: n={n_con}, 点击={x_con}, CTR={p_con:.4f}")
print(f"实验组: n={n_exp}, 点击={x_exp}, CTR={p_exp:.4f}")
print(f"Z 统计量: {z_stat:.4f}")
print(f"p 值（单尾）: {p_value:.6f}")
print(f"结论: {'显著' if p_value < 0.05 else '不显著'}")

## Step 3: 计算 CTR 差异的 95% 置信区间

In [ ]:
# 差异的标准误
se_diff = np.sqrt(p_con * (1 - p_con) / n_con + p_exp * (1 - p_exp) / n_exp)
diff = p_exp - p_con
ci_lower = diff - 1.96 * se_diff
ci_upper = diff + 1.96 * se_diff

print(f"CTR 差异: {diff:.4f}")
print(f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")

## Step 4: 计算 Cohen's h 效应量

In [ ]:
# Cohen's h
cohens_h = 2 * (np.arcsin(np.sqrt(p_exp)) - np.arcsin(np.sqrt(p_con)))

print(f"Cohen's h: {cohens_h:.4f}")
if cohens_h < 0.2:
    print("效应量解释: 可忽略 (negligible)")
elif cohens_h < 0.5:
    print("效应量解释: 小效应 (small)")
elif cohens_h < 0.8:
    print("效应量解释: 中等效应 (medium)")
else:
    print("效应量解释: 大效应 (large)")

## Step 5: 统计功效分析

In [ ]:
try:
    from statsmodels.stats.power import zt_ind_solve_power
    from statsmodels.stats.proportion import proportion_effectsize
    
    effect_size = proportion_effectsize(p_exp, p_con)
    power = zt_ind_solve_power(
        effect_size=effect_size,
        nobs1=n_exp,
        alpha=0.05,
        ratio=n_con/n_exp,
        alternative='larger'
    )
    print(f"统计功效: {power:.4f}")
except Exception as e:
    print(f"功效计算失败: {e}")
    power = None

## Step 6: 保存检验结果

In [ ]:
results = pd.DataFrame([{
    'n_con': n_con,
    'n_exp': n_exp,
    'p_con': p_con,
    'p_exp': p_exp,
    'z_stat': z_stat,
    'p_value': p_value,
    'ci_lower': ci_lower,
    'ci_upper': ci_upper,
    'cohens_h': cohens_h,
    'power': power if power else 0
}])

output.save_table(results, "hypothesis_test_results", chapter_prefix="ch03")
print("检验结果已保存")